# Step 1 — Ingest & Chunk
Loads Banking Analytics Bundle + Demographics Data Bundle from Snowflake Marketplace,
unions them into RAW_DOCUMENTS, then chunks with CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER.

**Constraints:** All data from Marketplace listings only. No external APIs.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd, warnings
warnings.filterwarnings("ignore")

session = get_active_session()

TARGET_DB       = "RAG_HACKATHON_DB"
TARGET_SCHEMA   = "RAG_SCHEMA"
TARGET_WH       = "SYSTEM$STREAMLIT_NOTEBOOK_WH"

BANKING_DB      = "BANKING_ANALYTICS_BUNDLE"
DEMOGRAPHICS_DB = "DEMOGRAPHICS_DATA_BUNDLE"

CHUNK_SIZE      = 512
CHUNK_OVERLAP   = 50
MIN_TEXT_LEN    = 80

session.sql(f"USE WAREHOUSE {TARGET_WH}").collect()
session.sql(f"CREATE DATABASE IF NOT EXISTS {TARGET_DB}").collect()
session.sql(f"USE DATABASE {TARGET_DB}").collect()
session.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}").collect()
session.sql(f"USE SCHEMA {TARGET_SCHEMA}").collect()

print("Session ready:", session.get_current_account())
print("BANKING_DB =", BANKING_DB)
print("DEMOGRAPHICS_DB =", DEMOGRAPHICS_DB)

In [ ]:
# CELL 2 — Find exact Marketplace database names
session.sql("SHOW DATABASES").collect()

dbs = session.sql("""
SELECT "name", "origin"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY 1
""").collect()

print("Available databases:\n")
for r in dbs:
    print(f"{r['name']:40} | origin: {r['origin']}")

print("\nUsing:")
print("BANKING_DB      =", BANKING_DB)
print("DEMOGRAPHICS_DB =", DEMOGRAPHICS_DB)

In [ ]:
# CELL 3 — Final selected source views for Step 1 ingestion

BANKING_VIEWS = [
    "FDIC",
    "FDIC_INSTITUTIONS",
    "FRED_FINANCIAL_LABOR_PERFORMANCE",
    "FRED_INTEREST_RATE_DATA",
    "FRED_UNEMPLOYEMENT_RATE"
]

DEMOGRAPHIC_VIEWS = [
    "DEMOGRAPHIC_STATS",
    "HOUSING_AND_FAMILIES",
    "NUMBER_BY_HOUSING_UNITS",
    "POPULATION_BY_GENDER_AND_AGE",
    "POPULATION_BY_RACE",
    "POPULATION_BY_URBAN_AND_RURAL",
    "POVERTY_STATUS_IN_12_MONTHS",
    "SELECTED_ECONOMIC_CHARACTERISTICS",
    "SELECTED_SOCIAL_CHARACTERISTICS"
]

print("Banking views selected:")
for v in BANKING_VIEWS:
    print(" -", f"{BANKING_DB}.INSIGHTS.{v}")

print("\nDemographics views selected:")
for v in DEMOGRAPHIC_VIEWS:
    print(" -", f"{DEMOGRAPHICS_DB}.INSIGHTS.{v}")

In [ ]:
# CELL 4 — Inspect selected views (columns + sample rows)

def inspect_view(db, schema_name, view_name, preview_rows=3):
    full_name = f"{db}.{schema_name}.{view_name}"
    print("\n" + "=" * 100)
    print("VIEW:", full_name)
    print("=" * 100)

    # Columns
    try:
        cols = session.sql(f"""
            SELECT column_name, data_type
            FROM {db}.information_schema.columns
            WHERE table_schema = '{schema_name}'
              AND table_name = '{view_name}'
            ORDER BY ordinal_position
        """).collect()

        print("\nColumns:")
        if cols:
            for c in cols:
                print(f" - {c['COLUMN_NAME']} ({c['DATA_TYPE']})")
        else:
            print("No columns found")
    except Exception as e:
        print("Could not inspect columns:", e)

    # Sample rows
    try:
        rows = session.sql(f"SELECT * FROM {full_name} LIMIT {preview_rows}").collect()
        print(f"\nSample rows ({preview_rows}):")
        if rows:
            for i, r in enumerate(rows, 1):
                print(f"\nRow {i}:")
                print(r.as_dict())
        else:
            print("No rows found")
    except Exception as e:
        print("Could not preview rows:", e)


print("\n########## BANKING VIEWS ##########")
for v in BANKING_VIEWS:
    inspect_view(BANKING_DB, "INSIGHTS", v, preview_rows=2)

print("\n########## DEMOGRAPHICS VIEWS ##########")
for v in DEMOGRAPHIC_VIEWS:
    inspect_view(DEMOGRAPHICS_DB, "INSIGHTS", v, preview_rows=2)

In [ ]:
# PRE-CELL-5 INSPECTION — gather all metadata needed for precision-optimized corpus

def inspect_columns(db, schema_name, object_name):
    print("\n" + "=" * 110)
    print(f"COLUMNS: {db}.{schema_name}.{object_name}")
    print("=" * 110)
    try:
        rows = session.sql(f"""
            SELECT column_name, data_type
            FROM {db}.information_schema.columns
            WHERE table_schema = '{schema_name}'
              AND table_name = '{object_name}'
            ORDER BY ordinal_position
        """).collect()

        if not rows:
            print("No columns found")
            return

        for r in rows:
            print(f"{r['COLUMN_NAME']:45} {r['DATA_TYPE']}")
    except Exception as e:
        print("Could not inspect columns:", e)


def preview_object(db, schema_name, object_name, n=2):
    print("\n" + "-" * 110)
    print(f"SAMPLE ROWS: {db}.{schema_name}.{object_name}")
    print("-" * 110)
    try:
        rows = session.sql(f"SELECT * FROM {db}.{schema_name}.{object_name} LIMIT {n}").collect()
        if not rows:
            print("No rows found")
            return
        for i, r in enumerate(rows, 1):
            print(f"\nRow {i}:")
            print(r.as_dict())
    except Exception as e:
        print("Could not preview rows:", e)


# Objects to inspect
objects_to_check = [
    (BANKING_DB,      "INSIGHTS", "FDIC"),
    (DEMOGRAPHICS_DB, "INSIGHTS", "SELECTED_ECONOMIC_CHARACTERISTICS"),
    (DEMOGRAPHICS_DB, "INSIGHTS", "POVERTY_STATUS_IN_12_MONTHS"),
    (DEMOGRAPHICS_DB, "INSIGHTS", "SELECTED_SOCIAL_CHARACTERISTICS"),
    (DEMOGRAPHICS_DB, "INSIGHTS", "HOUSEHOLD_TYPE_RELATIONSHIP"),
]

for db, schema_name, object_name in objects_to_check:
    inspect_columns(db, schema_name, object_name)
    preview_object(db, schema_name, object_name, n=2)

print("\n" + "=" * 110)
print("CURRENT EVAL_SET")
print("=" * 110)
try:
    for i, item in enumerate(EVAL_SET, 1):
        print(f"{i:02d}. q={item['q']} | type={item['type']} | source={item.get('source')}")
except Exception as e:
    print("Could not print EVAL_SET:", e)

In [ ]:
# CELL 5 — Build RAW_DOCUMENTS (high-signal corpus upgrade)

session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.RAW_DOCUMENTS AS

-- ============================================================================
-- BANKING: FDIC bank profiles
-- ============================================================================

SELECT * FROM (
    SELECT
        'banking' AS source,
        MD5(CAST(CERTIFICATE_NUMBER AS VARCHAR) || 'FDIC_PROFILE') AS doc_id,
        COALESCE(NAME, 'Bank Record') AS title,
        'Bank profile. Institution name: ' || COALESCE(NAME, 'Unknown bank') ||
        '. City: ' || COALESCE(CITY, 'unknown city') ||
        '. State: ' || COALESCE(STATE_NAME, 'unknown state') ||
        '. County: ' || COALESCE(COUNTY, 'unknown county') ||
        '. Assets: ' || COALESCE(CAST(ASSET AS VARCHAR), 'unknown') ||
        '. Bank class: ' || COALESCE(BANK_CLASS, 'unknown') ||
        '. Active status: ' || COALESCE(CAST(ACTIVE AS VARCHAR), 'unknown') ||
        '. Metro area: ' || COALESCE(CBSA_METRO_NAME, 'unknown') ||
        '. Failed bank list flag: ' || COALESCE(CAST(FAILED_BANKLIST AS VARCHAR), 'unknown') || '.' AS content
    FROM {BANKING_DB}.INSIGHTS.FDIC
    WHERE NAME IS NOT NULL
      AND CITY IS NOT NULL
      AND STATE_NAME IS NOT NULL
    LIMIT 2500
)

UNION ALL

-- ============================================================================
-- BANKING: Treasury interest rates / yield curve
-- ============================================================================

SELECT * FROM (
    SELECT
        'banking' AS source,
        MD5(CAST(DATE AS VARCHAR) || 'RATE_CURVE') AS doc_id,
        'Interest Rate and Yield Curve on ' || CAST(DATE AS VARCHAR) AS title,
        'Interest rate data on ' || CAST(DATE AS VARCHAR) ||
        '. Treasury yield curve information: 1-month yield ' || COALESCE(CAST(DAILY_TREASURY_YIELD_1_MONTH AS VARCHAR), 'unknown') ||
        ', 3-month yield ' || COALESCE(CAST(DAILY_TREASURY_YIELD_3_MONTH AS VARCHAR), 'unknown') ||
        ', 2-year yield ' || COALESCE(CAST(DAILY_TREASURY_YIELD_2_YEAR AS VARCHAR), 'unknown') ||
        ', 5-year yield ' || COALESCE(CAST(DAILY_TREASURY_YIELD_5_YEAR AS VARCHAR), 'unknown') ||
        ', 10-year yield ' || COALESCE(CAST(DAILY_TREASURY_YIELD_10_YEAR AS VARCHAR), 'unknown') ||
        ', and 30-year yield ' || COALESCE(CAST(DAILY_TREASURY_YIELD_30_YEAR AS VARCHAR), 'unknown') ||
        '. This record is relevant to interest rate trends, bond markets, yield curve shape, and banking conditions.' AS content
    FROM {BANKING_DB}.INSIGHTS.FRED_INTEREST_RATE_DATA
    WHERE DATE IS NOT NULL
      AND DAILY_TREASURY_YIELD_10_YEAR IS NOT NULL
      AND DAILY_TREASURY_YIELD_2_YEAR IS NOT NULL
      AND DAILY_TREASURY_YIELD_30_YEAR IS NOT NULL
    LIMIT 2500
)

UNION ALL

-- ============================================================================
-- BANKING: Macroeconomic / credit-risk indicators
-- ============================================================================

SELECT * FROM (
    SELECT
        'banking' AS source,
        MD5(CAST(OBSERVATION_DATE AS VARCHAR) || 'MACRO_CREDIT') AS doc_id,
        'Macroeconomic and Credit Risk Indicators on ' || CAST(OBSERVATION_DATE AS VARCHAR) AS title,
        'Macroeconomic indicators on ' || CAST(OBSERVATION_DATE AS VARCHAR) ||
        '. Unemployment rate seasonally adjusted: ' || COALESCE(CAST(UNEMPLOYMENT_RATE_SEASONALLY_ADJUSTED AS VARCHAR), 'unknown') ||
        '. Consumer price index seasonally adjusted: ' || COALESCE(CAST(CONSUMER_PRICE_INDEX_SEASONALLY_ADJUSTED AS VARCHAR), 'unknown') ||
        '. Federal funds rate: ' || COALESCE(CAST(FEDERAL_FUNDS AS VARCHAR), 'unknown') ||
        '. Money supply M2 seasonally adjusted: ' || COALESCE(CAST(M2_SEASONALLY_ADJUSTED AS VARCHAR), 'unknown') ||
        '. Treasury securities: ' || COALESCE(CAST(TREASURY_SECURITIES AS VARCHAR), 'unknown') ||
        '. This record is relevant to credit risk, lending conditions, inflation, employment, and monetary policy.' AS content
    FROM {BANKING_DB}.INSIGHTS.FRED_UNEMPLOYEMENT_RATE
    WHERE OBSERVATION_DATE IS NOT NULL
      AND (
            UNEMPLOYMENT_RATE_SEASONALLY_ADJUSTED IS NOT NULL
         OR CONSUMER_PRICE_INDEX_SEASONALLY_ADJUSTED IS NOT NULL
         OR FEDERAL_FUNDS IS NOT NULL
         OR M2_SEASONALLY_ADJUSTED IS NOT NULL
      )
    LIMIT 2500
)

UNION ALL

-- ============================================================================
-- BANKING: Labor and savings indicators
-- ============================================================================

SELECT * FROM (
    SELECT
        'banking' AS source,
        MD5(CAST(OBSERVATION_DATE AS VARCHAR) || 'LABOR_SAVE') AS doc_id,
        'Labor and Savings Indicators on ' || CAST(OBSERVATION_DATE AS VARCHAR) AS title,
        'Economic and labor performance on ' || CAST(OBSERVATION_DATE AS VARCHAR) ||
        '. Industrial production seasonally adjusted: ' || COALESCE(CAST(INDUSTRIAL_PRODUCTION_SEASONALLY_ADJUSTED AS VARCHAR), 'unknown') ||
        '. Total nonfarm employees seasonally adjusted: ' || COALESCE(CAST(ALL_EMPLOYEES_TOTAL_NONFARM_SEASONALLY_ADJUSTED AS VARCHAR), 'unknown') ||
        '. Labor force participation rate seasonally adjusted: ' || COALESCE(CAST(LABOR_FORCE_PARTICIPATION_RATE_SEASONALLY_ADJUSTED AS VARCHAR), 'unknown') ||
        '. Personal saving rate: ' || COALESCE(CAST(PERSONAL_SAVING_RATE AS VARCHAR), 'unknown') ||
        '. This record is relevant to labor markets, household savings, employment trends, and economic activity.' AS content
    FROM {BANKING_DB}.INSIGHTS.FRED_FINANCIAL_LABOR_PERFORMANCE
    WHERE OBSERVATION_DATE IS NOT NULL
      AND (
            ALL_EMPLOYEES_TOTAL_NONFARM_SEASONALLY_ADJUSTED IS NOT NULL
         OR LABOR_FORCE_PARTICIPATION_RATE_SEASONALLY_ADJUSTED IS NOT NULL
         OR PERSONAL_SAVING_RATE IS NOT NULL
         OR INDUSTRIAL_PRODUCTION_SEASONALLY_ADJUSTED IS NOT NULL
      )
    LIMIT 2500
)

UNION ALL

-- ============================================================================
-- DEMOGRAPHICS: State overview
-- ============================================================================

SELECT * FROM (
    SELECT
        'demographics' AS source,
        MD5(STATE || 'DEMO_OVERVIEW') AS doc_id,
        'Demographic Overview of ' || STATE AS title,
        'State demographic overview for ' || STATE ||
        '. Total population: ' || COALESCE(CAST(TOTAL_POPULATION AS VARCHAR), 'unknown') ||
        '. Male population: ' || COALESCE(CAST(MALE_COUNT AS VARCHAR), 'unknown') ||
        '. Female population: ' || COALESCE(CAST(FEMALE_COUNT AS VARCHAR), 'unknown') ||
        '. Urban population: ' || COALESCE(CAST(URBAN AS VARCHAR), 'unknown') ||
        '. Rural population: ' || COALESCE(CAST(RURAL AS VARCHAR), 'unknown') ||
        '. This record is relevant to population distribution, gender distribution, and urban versus rural composition.' AS content
    FROM {DEMOGRAPHICS_DB}.INSIGHTS.DEMOGRAPHIC_STATS
    WHERE STATE IS NOT NULL
)

UNION ALL

-- ============================================================================
-- DEMOGRAPHICS: Age structure
-- ============================================================================

SELECT * FROM (
    SELECT
        'demographics' AS source,
        MD5(STATE || 'DEMO_AGE') AS doc_id,
        'Age Structure of ' || STATE AS title,
        'Age distribution for ' || STATE ||
        '. Male under age 5: ' || COALESCE(CAST(AGE_UNDER_5_M AS VARCHAR), 'unknown') ||
        '. Female under age 5: ' || COALESCE(CAST(AGE_UNDER_5_F AS VARCHAR), 'unknown') ||
        '. Male age 18 to 19: ' || COALESCE(CAST(AGE_18_AND_19_M AS VARCHAR), 'unknown') ||
        '. Female age 18 to 19: ' || COALESCE(CAST(AGE_18_AND_19_F AS VARCHAR), 'unknown') ||
        '. Male age 25 to 29: ' || COALESCE(CAST(AGE_25_TO_29_M AS VARCHAR), 'unknown') ||
        '. Female age 25 to 29: ' || COALESCE(CAST(AGE_25_TO_29_F AS VARCHAR), 'unknown') ||
        '. Male age 65 to 66: ' || COALESCE(CAST(AGE_65_AND_66_M AS VARCHAR), 'unknown') ||
        '. Female age 65 to 66: ' || COALESCE(CAST(AGE_65_AND_66_F AS VARCHAR), 'unknown') ||
        '. Male age 85 and over: ' || COALESCE(CAST(AGE_85_AND_OVER_M AS VARCHAR), 'unknown') ||
        '. Female age 85 and over: ' || COALESCE(CAST(AGE_85_AND_OVER_F AS VARCHAR), 'unknown') ||
        '. This record is relevant to age groups, youth population, working age population, and senior population.' AS content
    FROM {DEMOGRAPHICS_DB}.INSIGHTS.DEMOGRAPHIC_STATS
    WHERE STATE IS NOT NULL
)

UNION ALL

-- ============================================================================
-- DEMOGRAPHICS: Race and ethnicity
-- ============================================================================

SELECT * FROM (
    SELECT
        'demographics' AS source,
        MD5(STATE || 'DEMO_RACE') AS doc_id,
        'Race and Ethnicity Profile of ' || STATE AS title,
        'Race and ethnicity distribution for ' || STATE ||
        '. Hispanic or Latino population: ' || COALESCE(CAST(HISPANIC_OR_LATINO AS VARCHAR), 'unknown') ||
        '. White alone population: ' || COALESCE(CAST(WHITE_ALONE AS VARCHAR), 'unknown') ||
        '. Black or African American alone population: ' || COALESCE(CAST(BLACK_OR_AFRICAN_AMERICAN_ALONE AS VARCHAR), 'unknown') ||
        '. Asian alone population: ' || COALESCE(CAST(ASIAN_ALONE AS VARCHAR), 'unknown') ||
        '. American Indian and Alaska Native alone population: ' || COALESCE(CAST(AMERICAN_INDIAN_AND_ALASKA_NATIVE_ALONE AS VARCHAR), 'unknown') ||
        '. Native Hawaiian and other Pacific Islander alone population: ' || COALESCE(CAST(NATIVE_HAWAIIAN_AND_OTHER_PACIFIC_ISLANDER_ALONE AS VARCHAR), 'unknown') ||
        '. Two or more races population: ' || COALESCE(CAST(TWO_OR_MORE_RACES AS VARCHAR), 'unknown') ||
        '. This record is relevant to race, ethnicity, and population composition.' AS content
    FROM {DEMOGRAPHICS_DB}.INSIGHTS.DEMOGRAPHIC_STATS
    WHERE STATE IS NOT NULL
)
""").collect()

rows = session.sql(f"""
SELECT source,
       COUNT(*) AS docs,
       ROUND(AVG(LENGTH(content))) AS avg_chars
FROM {TARGET_DB}.{TARGET_SCHEMA}.RAW_DOCUMENTS
GROUP BY source
ORDER BY source
""").collect()

print("RAW_DOCUMENTS created (UPGRADED)")
for r in rows:
    print(r)

In [ ]:
# CELL 6 — Chunk with CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER
session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS AS
WITH ex AS (
  SELECT source,doc_id,title,
         SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(content,'none',{CHUNK_SIZE},{CHUNK_OVERLAP}) AS chunks
  FROM {TARGET_DB}.{TARGET_SCHEMA}.RAW_DOCUMENTS
)
SELECT source,doc_id,title,c.index::INT AS chunk_index,c.value::VARCHAR AS chunk_text,
       LENGTH(c.value::VARCHAR) AS chunk_len_chars
FROM ex, LATERAL FLATTEN(INPUT=>chunks) c
WHERE LENGTH(c.value::VARCHAR)>50
""").collect()
s=session.sql(f"SELECT source,COUNT(*) chunks,COUNT(DISTINCT doc_id) docs,ROUND(AVG(chunk_len_chars)) avg_chars FROM {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS GROUP BY source").to_pandas()
print('CHUNKED_DOCUMENTS created'); print(s.to_string(index=False))

In [ ]:
# CELL 7 — Validate
sample=session.sql(f"SELECT source,title,chunk_index,LEFT(chunk_text,300) preview FROM (SELECT *,ROW_NUMBER() OVER (PARTITION BY source ORDER BY RANDOM()) rn FROM {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS) WHERE rn<=2 ORDER BY source,chunk_index").to_pandas()
for _,r in sample.iterrows():
    print(f"\n[{r['SOURCE']}] {str(r['TITLE'])[:55]} (chunk {r['CHUNK_INDEX']}):")
    print(f"  {r['PREVIEW']}...")
print('\nStep 1 COMPLETE — CHUNKED_DOCUMENTS ready for Step 2')

## Corpus Enrichment
Fixes low retrieval precision by:
1. Adding dataset-level summary chunks (schema descriptions, column inventories, coverage stats)
2. Ingesting demographics mapped views (poverty, economic, social, housing categories)
3. Sampling banking to reduce duplicate bank-profile noise
4. Rebuilding CHUNKED_DOCUMENTS with balanced data

In [ ]:
session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.SUMMARY_CHUNKS (
    source      VARCHAR,
    doc_id      VARCHAR,
    title       VARCHAR,
    chunk_index INT DEFAULT 0,
    chunk_text  VARCHAR,
    chunk_len_chars INT
)
""").collect()

summaries = [
    ('banking', 'summary-fdic', 'FDIC Banking Institutions Overview',
     'The FDIC dataset contains information about FDIC-insured banking institutions across the United States. '
     'Key fields include: institution name (NAME), city (CITY), state (STATE_NAME), county (COUNTY), '
     'total assets (ASSET), bank classification (BANK_CLASS such as NM=National Member, SM=State Member), '
     'active status (ACTIVE), address (ADDRESS), ZIP code (ZIP), CBSA metro area names, '
     'and failed bank indicators (FAILED_BANKLIST, FAILED_BANK_CLOSING_DATE, FAILED_BANK_ACQUIRING_INSITUTION). '
     'The dataset covers all 50 US states plus territories. It tracks both active and failed institutions. '
     'There are approximately 4,486 institution records. '
     'Bank classes include NM (National Member), SM (State Member), and others indicating regulatory classification.'),

    ('banking', 'summary-fdic-institutions', 'FDIC Institutions Detailed View Overview',
     'The FDIC_INSTITUTIONS view provides detailed regulatory and geographic data about banking institutions. '
     'It contains approximately 67,922 records including branch-level information. '
     'Key fields include: certificate numbers (CERT, ULTCERT), institution names (INSTNAME, HCTMULT), '
     'geographic identifiers (CBSA_NO, CBSA_METRO_NAME, CSA_NO, MSA, TRACT, FIPS codes), '
     'regulatory info (CHRTAGNT, REGAGENT2, INSAGNT1, FDICDBS, FDICREGN, SUPRV_FD, SPECGRP), '
     'financial data (ASSET, DEPDOM, NETINC, ROA, ROE), and temporal fields (ENDEFYMD, RUNDATE, RISESSION). '
     'This view links institutions to their metro/micropolitan statistical areas and Census tracts.'),

    ('banking', 'summary-fred-unemployment', 'FRED Unemployment and Economic Indicators Overview',
     'The FRED_UNEMPLOYEMENT_RATE view contains macroeconomic time-series data from the Federal Reserve. '
     'Key metrics include: federal funds rate (FEDERAL_FUNDS), unemployment rate both seasonally and '
     'not seasonally adjusted (UNEMPLOYMENT_RATE_SEASONALLY_ADJUSTED, UNEMPLOYMENT_RATE_NOT_SEASONALLY_ADJUSTED), '
     'Consumer Price Index (CONSUMER_PRICE_INDEX_SEASONALLY_ADJUSTED, CONSUMER_PRICE_INDEX_NOT_SEASONALLY_ADJUSTED), '
     'M2 money supply (M2_SEASONALLY_ADJUSTED, M2_NOT_SEASONALLY_ADJUSTED), '
     'two-year Treasury constant maturity yield (TWO_YEAR_TREASURY_CONSTANT), '
     'and Treasury securities yield (TREASURY_SECURITIES). Data is indexed by OBSERVATION_DATE.'),

    ('banking', 'summary-fred-labor', 'FRED Financial and Labor Performance Overview',
     'The FRED_FINANCIAL_LABOR_PERFORMANCE view tracks labor market and industrial production metrics over time. '
     'It contains approximately 1,274 monthly observations indexed by OBSERVATION_DATE. '
     'Key metrics: industrial production both seasonally and not seasonally adjusted '
     '(INDUSTRIAL_PRODUCTION_SEASONALLY_ADJUSTED, INDUSTRIAL_PRODUCTION_NOT_SEASONALLY_ADJUSTED), '
     'total nonfarm employment (ALL_EMPLOYEES_TOTAL_NONFARM_SEASONALLY_ADJUSTED, ALL_EMPLOYEES_TOTAL_NONFARM_NOT_SEASONALLY_ADJUSTED), '
     'labor force participation rate (LABOR_FORCE_PARTICIPATION_RATE_SEASONALLY_ADJUSTED, '
     'LABOR_FORCE_PARTICIPATION_RATE_NOT_SEASONALLY_ADJUSTED), and personal saving rate (PERSONAL_SAVING_RATE). '
     'This data helps analyze employment trends, industrial output, and savings behavior over decades.'),

    ('banking', 'summary-fred-interest', 'FRED Interest Rate and Treasury Yield Curve Overview',
     'The FRED_INTEREST_RATE_DATA view contains daily Treasury yield curve rates and bill rates. '
     'It is a large dataset with over 466 million rows indexed by DATE. '
     'Key fields include: Daily Treasury Par Yield Curve Rates for maturities from 1 month to 30 years '
     '(DAILY_TREASURY_YIELD_1_MONTH through DAILY_TREASURY_YIELD_30_YEAR), '
     'Daily Real Treasury Yield Curve Rates for 5 to 30 year maturities, '
     'Treasury Bill Rates at various durations (4-week through 52-week) for both discount and coupon equivalent, '
     'and long-term composites (LONG_TERM_TREASURY_COMPOSITE_RATE_GT_10_YEARS, LONG_TERM_TREASURY_20_YEAR_CMT). '
     'This data tracks interest rate trends and yield curve movements over time.'),

    ('demographics', 'summary-demographic-stats', 'Demographic Statistics Overview',
     'The DEMOGRAPHIC_STATS view provides comprehensive population statistics for all 52 US states and territories. '
     'Key fields include: total population (TOTAL_POPULATION), gender breakdown (MALE_COUNT, FEMALE_COUNT), '
     'detailed age distribution by gender (AGE_UNDER_5 through AGE_85_AND_OVER for both M and F), '
     'race and ethnicity data (WHITE_ALONE, BLACK_OR_AFRICAN_AMERICAN_ALONE, ASIAN_ALONE, '
     'AMERICAN_INDIAN_AND_ALASKA_NATIVE_ALONE, NATIVE_HAWAIIAN_AND_OTHER_PACIFIC_ISLANDER_ALONE, '
     'TWO_OR_MORE_RACES, SOME_OTHER_RACE_ALONE), '
     'Hispanic/Latino breakdown (HISPANIC_OR_LATINO with race sub-categories, NOT_HISPANIC_OR_LATINO), '
     'and urban/rural classification (URBAN, RURAL). Data is organized by STATE.'),

    ('demographics', 'summary-poverty', 'Poverty Status in 12 Months Overview',
     'The POVERTY_STATUS_IN_12_MONTHS and its MAPPED view track poverty statistics across US states. '
     'The mapped view contains approximately 8,424 records organized by STATE, GEO_ID, and hierarchical categories '
     '(CATEGORY_LEVEL1 through CATEGORY_LEVEL4). '
     'Categories include: poverty status determinations, employment status breakdowns for those in poverty, '
     'age group analysis (under 18, 18-64, 65+), gender breakdowns, race/ethnicity poverty rates, '
     'educational attainment and poverty, work experience and poverty status, '
     'and unrelated individuals poverty analysis. '
     'Values include ESTIMATE (count or percentage) and MARGIN_OF_ERROR.'),

    ('demographics', 'summary-economic-chars', 'Selected Economic Characteristics Overview',
     'The SELECTED_ECONOMIC_CHARACTERISTICS and its MAPPED view contain economic data for all US states. '
     'The mapped view has approximately 6,864 records organized by STATE, GEO_ID, '
     'and hierarchical categories (CATEGORY_LEVEL1 through CATEGORY_LEVEL4). '
     'Major categories include: EMPLOYMENT_STATUS (labor force participation, unemployment), '
     'COMMUTING_TO_WORK (transportation modes, travel time), '
     'OCCUPATION (management, service, sales, production, etc.), '
     'INDUSTRY (agriculture, construction, manufacturing, retail, healthcare, education, etc.), '
     'CLASS_OF_WORKER (private, government, self-employed), '
     'INCOME_AND_BENEFITS (household income brackets, median income, social security, retirement), '
     'and HEALTH_INSURANCE_COVERAGE. '
     'Values include ESTIMATE counts and PERCENT with margins of error.'),

    ('demographics', 'summary-social-chars', 'Selected Social Characteristics Overview',
     'The SELECTED_SOCIAL_CHARACTERISTICS and its MAPPED view contain social indicators for all US states. '
     'The mapped view has approximately 7,497 records organized by STATE, GEO_ID, '
     'and hierarchical categories. '
     'Major categories include: EDUCATIONAL_ATTAINMENT (high school, bachelors, graduate degrees), '
     'MARITAL_STATUS (married, widowed, divorced, never married), '
     'FERTILITY (birth rates), GRANDPARENTS (caregiving), '
     'VETERAN_STATUS, DISABILITY_STATUS, '
     'PLACE_OF_BIRTH (native vs foreign born), CITIZENSHIP_STATUS, '
     'LANGUAGE_SPOKEN_AT_HOME (English, Spanish, Asian languages, etc.), '
     'and ANCESTRY (American, Irish, German, English, Italian, etc.). '
     'These social factors help understand community composition and needs.'),

    ('demographics', 'summary-housing', 'Housing Characteristics Overview',
     'The SELECETD_HOUSING_CHARACTERISTICS and its MAPPED view contain housing data for all US states. '
     'The mapped view has approximately 7,436 records organized by STATE, GEO_ID, '
     'and hierarchical categories. '
     'Major categories include: HOUSING_OCCUPANCY (occupied vs vacant units), '
     'UNITS_IN_STRUCTURE (detached, attached, apartments by size), '
     'YEAR_STRUCTURE_BUILT (housing age distribution), '
     'ROOMS and BEDROOMS counts, '
     'HOUSING_TENURE (owner-occupied vs renter-occupied), '
     'YEAR_HOUSEHOLDER_MOVED_INTO_UNIT, '
     'VEHICLES_AVAILABLE, HOUSE_HEATING_FUEL (gas, electric, oil, wood, solar), '
     'SELECTED_MONTHLY_OWNER_COSTS, GROSS_RENT, and RENT_AS_PERCENT_OF_INCOME. '
     'This data tracks housing conditions, costs, and characteristics across states.'),

    ('demographics', 'summary-housing-families', 'Housing and Families Overview',
     'The HOUSING_AND_FAMILIES and its MAPPED view contain household composition data for all US states. '
     'The mapped view has approximately 4,160 records organized by STATE, GEO_ID, '
     'and hierarchical categories. '
     'Categories include: HOUSEHOLD_TYPE (family vs nonfamily households), '
     'RELATIONSHIP_TO_HOUSEHOLDER (spouse, child, other relative, nonrelative), '
     'MARITAL_STATUS within households, '
     'FAMILY_TYPE (married couple, male householder no wife, female householder no husband), '
     'and CHILDREN_UNDER_18 presence. '
     'This data helps understand household size, family structures, and living arrangements across states.'),

    ('demographics', 'summary-population', 'Population Demographics Overview',
     'Multiple population views provide breakdowns by: '
     'GENDER (POPULATION_BY_GENDER with MALE_COUNT, FEMALE_COUNT per state), '
     'RACE (POPULATION_BY_RACE with WHITE, BLACK, ASIAN, NATIVE_HAWAIIAN, AMERICAN_INDIAN, TWO_OR_MORE categories), '
     'ETHNICITY (POPULATION_BY_ETHINICITY and BY_ETHINICITY_AND_RACE with HISPANIC_OR_LATINO breakdowns), '
     'URBAN/RURAL (POPULATION_BY_URBAN_AND_RURAL with URBAN and RURAL counts per state), '
     'and GENDER_AND_AGE (detailed age brackets by gender). '
     'All views cover 52 US states and territories. '
     'The HOUSEHOLD_TYPE_RELATIONSHIP view tracks household composition and living arrangements. '
     'NUMBER_BY_HOUSING_UNITS tracks housing unit counts per state.'),
]

for src, doc_id, title, text in summaries:
    text_escaped = text.replace("'", "''")
    title_escaped = title.replace("'", "''")
    session.sql(f"""
        INSERT INTO {TARGET_DB}.{TARGET_SCHEMA}.SUMMARY_CHUNKS
            (source, doc_id, title, chunk_index, chunk_text, chunk_len_chars)
        VALUES ('{src}', '{doc_id}', '{title_escaped}', 0, '{text_escaped}', {len(text)})
    """).collect()

count = session.sql(f"SELECT COUNT(*) AS cnt FROM {TARGET_DB}.{TARGET_SCHEMA}.SUMMARY_CHUNKS").collect()[0]['CNT']
print(f'Inserted {count} summary chunks')

In [ ]:
session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.DEMOGRAPHICS_MAPPED_RAW (
    source      VARCHAR DEFAULT 'demographics',
    doc_id      VARCHAR,
    title       VARCHAR,
    content     VARCHAR
)
""").collect()

mapped_views = [
    ('POVERTY_STATUS_IN_12_MONTHS_MAPPED', 'poverty', True),
    ('SELECTED_ECONOMIC_CHARACTERISTICS_MAPPED', 'economic', True),
    ('SELECTED_SOCIAL_CHARACTERISTICS_MAPPED', 'social', True),
    ('SELECETD_HOUSING_CHARACTERISTICS_MAPPED', 'housing', False),
    ('HOUSING_AND_FAMILIES_MAPPED', 'housing-families', True),
]

for view_name, label, has_level4 in mapped_views:
    print(f'Ingesting {view_name}...')
    level4_clause = "|| CASE WHEN CATEGORY_LEVEL4 IS NOT NULL THEN ' > ' || CATEGORY_LEVEL4 ELSE '' END" if has_level4 else ""
    session.sql(f"""
        INSERT INTO {TARGET_DB}.{TARGET_SCHEMA}.DEMOGRAPHICS_MAPPED_RAW (doc_id, title, content)
        SELECT
            '{label}-' || STATE || '-' || ROW_NUMBER() OVER (PARTITION BY STATE ORDER BY CATEGORY_LEVEL1, CATEGORY_LEVEL2),
            COALESCE(CATEGORY_LEVEL1, 'General') || ' in ' || STATE,
            '{label} data for ' || STATE || '. '
            || 'Category: ' || COALESCE(CATEGORY_LEVEL1, 'N/A')
            || CASE WHEN CATEGORY_LEVEL2 IS NOT NULL THEN ' > ' || CATEGORY_LEVEL2 ELSE '' END
            || CASE WHEN CATEGORY_LEVEL3 IS NOT NULL THEN ' > ' || CATEGORY_LEVEL3 ELSE '' END
            {level4_clause}
            || '. Estimate: ' || COALESCE(CAST(ESTIMATE AS VARCHAR), 'N/A')
            || '. Margin of error: ' || COALESCE(CAST(MARGIN_OF_ERROR AS VARCHAR), 'N/A')
            || '.'
        FROM {DEMOGRAPHICS_DB}.INSIGHTS.{view_name}
    """).collect()

cnt = session.sql(f"SELECT COUNT(*) AS cnt FROM {TARGET_DB}.{TARGET_SCHEMA}.DEMOGRAPHICS_MAPPED_RAW").collect()[0]['CNT']
print(f'\nTotal demographics mapped rows ingested: {cnt}')

In [ ]:
print('Rebuilding CHUNKED_DOCUMENTS with enriched corpus...')

session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS AS

WITH existing_banking_sampled AS (
    SELECT source, doc_id, title, chunk_index, chunk_text, chunk_len_chars
    FROM {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS
    WHERE source = 'banking'
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY LEFT(title, 30)
        ORDER BY RANDOM()
    ) <= 2
),

existing_demographics AS (
    SELECT source, doc_id, title, chunk_index, chunk_text, chunk_len_chars
    FROM {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS
    WHERE source = 'demographics'
),

new_demo_chunked AS (
    SELECT source, doc_id, title,
           c.index::INT AS chunk_index,
           c.value::VARCHAR AS chunk_text,
           LENGTH(c.value::VARCHAR) AS chunk_len_chars
    FROM (
        SELECT source, doc_id, title,
               SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(content, 'none', {CHUNK_SIZE}, {CHUNK_OVERLAP}) AS chunks
        FROM {TARGET_DB}.{TARGET_SCHEMA}.DEMOGRAPHICS_MAPPED_RAW
    ) ex, LATERAL FLATTEN(INPUT => chunks) c
    WHERE LENGTH(c.value::VARCHAR) > 50
),

summary AS (
    SELECT source, doc_id, title, chunk_index, chunk_text, chunk_len_chars
    FROM {TARGET_DB}.{TARGET_SCHEMA}.SUMMARY_CHUNKS
)

SELECT * FROM existing_banking_sampled
UNION ALL
SELECT * FROM existing_demographics
UNION ALL
SELECT * FROM new_demo_chunked
UNION ALL
SELECT * FROM summary
""").collect()

stats = session.sql(f"""
    SELECT source, COUNT(*) AS chunks, COUNT(DISTINCT doc_id) AS docs,
           ROUND(AVG(LENGTH(chunk_text))) AS avg_len
    FROM {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS
    GROUP BY source
""").to_pandas()
print('Rebuilt CHUNKED_DOCUMENTS:')
print(stats.to_string(index=False))
total = session.sql(f"SELECT COUNT(*) AS cnt FROM {TARGET_DB}.{TARGET_SCHEMA}.CHUNKED_DOCUMENTS").collect()[0]['CNT']
print(f'\nTotal chunks: {total}')